# Phase 3: Deep learning model build

**Project:** SynText-Scanner  
**Goal:** Clean raw text data using NLP techniques (Lemmatization) and explore statistical differences between AI and Human-authored content.

In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments, BertForSequenceClassification
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [15]:
print ('Loading processed datasets...')

df_train = pd.read_csv('../data/processed data/train_cleaned.csv', index_col=0)
df_test = pd.read_csv('../data/processed data/test_cleaned.csv')

df_train = df_train.dropna(subset=['cleaned_text', 'label']).reset_index(drop=True)
df_test = df_test.dropna(subset=['cleaned_text', 'label']).reset_index(drop=True)
print(f"Data ready, train size: {len(df_train)}, test size: {len(df_test)}")

model_name = 'distilbert-base-uncased'
print(f"\nInitializing tokenizer from Hugging Face for '{model_name}'...")
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

sample_text = df_train['cleaned_text'].iloc[0]
print("\nTokennizer test:")
print(f"Original text (first 50 characters): '{sample_text[:50]}...'")

encoded_sample = tokenizer.encode(sample_text[:50])
print(f"Token IDs: {encoded_sample}")



Loading processed datasets...
Data ready, train size: 6069, test size: 1367

Initializing tokenizer from Hugging Face for 'distilbert-base-uncased'...

Tokennizer test:
Original text (first 50 characters): 'study investigate chemical composition thermal spr...'
Token IDs: [101, 2817, 8556, 5072, 5512, 9829, 11867, 2099, 102]


In [16]:
print("Starting batch tokenization...")

train_encodings = tokenizer(df_train['cleaned_text'].tolist(),
                            truncation=True,
                            padding=True,
                            max_length=256)

test_encodings = tokenizer(df_test['cleaned_text'].tolist(),
                           truncation=True,
                           padding=True,
                           max_length=256)
print("Bath tokenization complete.")

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)
    
print("Building PyTorch datasets...")

train_dataset = TextDataset(train_encodings, df_train['label'].tolist())
test_dataset = TextDataset(test_encodings, df_test['label'].tolist())

print("PyTorch datasets ready.")
print(f"Sample item keys: {list(train_dataset[0].keys())}")


Starting batch tokenization...
Bath tokenization complete.
Building PyTorch datasets...
PyTorch datasets ready.
Sample item keys: ['input_ids', 'attention_mask', 'labels']


In [10]:
print("Loading pre-trained DistilBERT model...")

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
print(f"Model loaded and moved to device: {device}")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

print("\nSetting up training arguments...")
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True
)

print("Initializing Hugging Face Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("\nSequence start. Fine-tuning the model...")
trainer.train()

print("\nTraining complete!")

Loading pre-trained DistilBERT model...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9517.58it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Model loaded and moved to device: cuda

Setting up training arguments...
Initializing Hugging Face Trainer...

Sequence start. Fine-tuning the model...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.031772,3.644455,0.494514,0.356011,0.489744,0.279649
2,0.022816,3.981044,0.493782,0.244541,0.480687,0.163982
3,0.002222,4.361619,0.490856,0.314961,0.480480,0.234261


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.26it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Training complete!


In [11]:
print("Loading a fresh pre-trained DistilBERT model for Reverse Testing...")

reverse_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
reverse_model.to(device)

print("\nSetting up training arguments for Reverse Testing...")
reverse_training_args = TrainingArguments(
    output_dir='./results_reverse',
    num_train_epochs=3,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=64,   
    warmup_steps=50,                 
    weight_decay=0.01,               
    logging_dir='./logs_reverse',
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

print("Initializing the Reverse Trainer...")
reverse_trainer = Trainer(
    model=reverse_model,
    args=reverse_training_args,
    train_dataset=test_dataset,
    eval_dataset=train_dataset,
    compute_metrics=compute_metrics
)

print("\nSequence start. Fine-tuning the Reverse Model...")
reverse_trainer.train()

Loading a fresh pre-trained DistilBERT model for Reverse Testing...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7692.16it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



Setting up training arguments for Reverse Testing...
Initializing the Reverse Trainer...

Sequence start. Fine-tuning the Reverse Model...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.692653,0.692742,0.505685,0.671701,0.505685,1.000000
2,0.696351,0.692897,0.534849,0.439102,0.562627,0.360052
3,0.695748,0.706260,0.363651,0.491708,0.412453,0.608667


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.15it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=258, training_loss=0.6942203248194022, metrics={'train_runtime': 41.172, 'train_samples_per_second': 99.607, 'train_steps_per_second': 6.266, 'total_flos': 271624400944128.0, 'train_loss': 0.6942203248194022, 'epoch': 3.0})

In [18]:
full_model_name = 'bert-base-uncased'

print(f"Initializing full-scale {full_model_name}...")
tokenizer_bert = BertTokenizer.from_pretrained(full_model_name)
model_bert = BertForSequenceClassification.from_pretrained(full_model_name, num_labels=2)
model_bert.to(device)

train_encodings_bert = tokenizer_bert(df_train['cleaned_text'].tolist(), truncation=True, padding=True, max_length=256)
test_encodings_bert = tokenizer_bert(df_test['cleaned_text'].tolist(), truncation=True, padding=True, max_length=256)

train_dataset_bert = TextDataset(train_encodings_bert, df_train['label'].tolist())
test_dataset_bert = TextDataset(test_encodings_bert, df_test['label'].tolist())

training_args_bert = TrainingArguments(
    output_dir='./results_bert',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True
)

trainer_bert = Trainer(
    model=model_bert,
    args=training_args_bert,
    train_dataset=train_dataset_bert,
    eval_dataset=test_dataset_bert,
    compute_metrics=compute_metrics
)

print(f"\Launching full BERT fine-tuning on {device}...")
trainer_bert.train()

Initializing full-scale bert-base-uncased...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8794.85it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

\Launching full BERT fine-tuning on cuda...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.035490,3.944825,0.487930,0.325626,0.476056,0.247438
2,0.011184,4.305312,0.493050,0.319921,0.485119,0.238653
3,0.000259,4.515992,0.494514,0.297050,0.486667,0.213763


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.05it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=1140, training_loss=0.02037685919887991, metrics={'train_runtime': 145.5923, 'train_samples_per_second': 125.055, 'train_steps_per_second': 7.83, 'total_flos': 2395231492469760.0, 'train_loss': 0.02037685919887991, 'epoch': 3.0})